# PS S6E6 — LightGBM v2

**Baseline OOF:** 0.96418 | **Baseline LB:** 0.96509  
**What's new:**
- Additional features: `log_redshift`, `redshift²`, redshift × color interactions, wide color `u_z`
- Optuna hyperparameter search (fast: 30% data sample, 3-fold, 50 trials)
- Final model trained with best params on full data, 5-fold CV

**Target:** Beat 0.96418 OOF, specifically improve STAR precision (was 0.89)

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
CFG = dict(
    n_folds         = 5,
    seed            = 42,
    # Optuna search
    optuna_trials   = 50,
    optuna_folds    = 3,
    optuna_sample   = 0.3,   # fraction of train used during search
    # Final model
    n_estimators    = 2000,
    early_stop      = 50,
)

BASELINE_OOF = 0.96418
BASELINE_LB  = 0.96509

## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/train.csv', index_col='id')
test  = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/test.csv',  index_col='id')

print(f'Train: {train.shape}  |  Test: {test.shape}')

## 3. Feature Engineering

New features on top of v1 color indices:
- `log_redshift` — log1p transform; better model resolution at the low end where STAR/GALAXY confusion is
- `redshift_sq` — captures non-linearity around the STAR boundary
- `redshift × g_r`, `redshift × u_g`, `redshift × g_i` — the STAR locus in color space shifts with redshift for galaxies/QSOs but not for stars
- `u_z` — widest color baseline across all filters

In [ ]:
def engineer_features(df):
    df = df.copy()

    # v1 color indices
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']

    # v2 additions
    df['u_z']          = df['u'] - df['z']                  # widest color baseline
    df['log_redshift'] = np.log1p(df['redshift'])            # compress skewed tail
    df['redshift_sq']  = df['redshift'] ** 2                 # non-linearity near zero
    df['rs_x_gr']      = df['redshift'] * df['g_r']          # color-redshift interaction
    df['rs_x_ug']      = df['redshift'] * df['u_g']
    df['rs_x_gi']      = df['redshift'] * df['g_i']

    return df

train = engineer_features(train)
test  = engineer_features(test)

V1_COLOR_COLS = ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'g_z']
V2_FEAT_COLS  = ['u_z', 'log_redshift', 'redshift_sq', 'rs_x_gr', 'rs_x_ug', 'rs_x_gi']

print(f'Total new features added: {len(V2_FEAT_COLS)}')
print(train[V2_FEAT_COLS].describe().round(3))

## 4. Preprocessing

In [ ]:
CAT_COLS  = ['spectral_type', 'galaxy_population']
NUM_COLS  = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
FEAT_COLS = NUM_COLS + V1_COLOR_COLS + V2_FEAT_COLS + CAT_COLS
TARGET    = 'class'

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[CAT_COLS] = oe.fit_transform(train[CAT_COLS])
test[CAT_COLS]  = oe.transform(test[CAT_COLS])

le = LabelEncoder()
y  = le.fit_transform(train[TARGET])
print('Class mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X      = train[FEAT_COLS]
X_test = test[FEAT_COLS]
print(f'\nFeature matrix: {X.shape}')

## 5. Optuna Hyperparameter Search

Run on a stratified 30% sample with 3-fold CV to keep wall time reasonable.  
We use `balanced_accuracy_score` as the objective since that's the competition metric.

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified sample for Optuna
_, X_opt, _, y_opt = train_test_split(
    X, y,
    test_size=CFG['optuna_sample'],
    stratify=y,
    random_state=CFG['seed'],
)
print(f'Optuna search set: {X_opt.shape}')

def objective(trial):
    params = dict(
        n_estimators      = 500,
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 63, 511),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 200),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        class_weight      = 'balanced',
        n_jobs            = -1,
        random_state      = CFG['seed'],
        verbose           = -1,
    )

    skf = StratifiedKFold(n_splits=CFG['optuna_folds'], shuffle=True, random_state=CFG['seed'])
    scores = []
    for tr_idx, val_idx in skf.split(X_opt, y_opt):
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_opt.iloc[tr_idx], y_opt[tr_idx],
            eval_set=[(X_opt.iloc[val_idx], y_opt[val_idx])],
            callbacks=[lgb.early_stopping(30, verbose=False)],
            categorical_feature=CAT_COLS,
        )
        preds = le.inverse_transform(m.predict(X_opt.iloc[val_idx]))
        truth = le.inverse_transform(y_opt[val_idx])
        scores.append(balanced_accuracy_score(truth, preds))

    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=CFG['seed']))
study.optimize(objective, n_trials=CFG['optuna_trials'], show_progress_bar=True)

print(f'\nBest trial score: {study.best_value:.5f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Optuna optimization history
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout()

In [ ]:
# Parameter importances
optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout()

## 6. Final Model — 5-Fold CV with Best Params

In [ ]:
BEST_PARAMS = dict(
    **study.best_params,
    n_estimators  = CFG['n_estimators'],
    class_weight  = 'balanced',
    n_jobs        = -1,
    random_state  = CFG['seed'],
    verbose       = -1,
)

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

oof_proba  = np.zeros((len(X), len(le.classes_)))
test_proba = np.zeros((len(X_test), len(le.classes_)))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = lgb.LGBMClassifier(**BEST_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(CFG['early_stop'], verbose=False),
            lgb.log_evaluation(200),
        ],
        categorical_feature=CAT_COLS,
    )

    val_proba = model.predict_proba(X_val)
    val_preds = le.inverse_transform(val_proba.argmax(axis=1))
    val_true  = le.inverse_transform(y_val)

    score = balanced_accuracy_score(val_true, val_preds)
    fold_scores.append(score)
    oof_proba[val_idx] = val_proba
    test_proba += model.predict_proba(X_test) / CFG['n_folds']

    print(f'  Fold {fold+1}/{CFG["n_folds"]} | best_iter={model.best_iteration_} | balanced_acc={score:.5f}')

print(f'\nCV mean: {np.mean(fold_scores):.5f}')
print(f'CV std:  {np.std(fold_scores):.5f}')

## 7. OOF Evaluation

In [ ]:
oof_preds_labels = le.inverse_transform(oof_proba.argmax(axis=1))
oof_true_labels  = le.inverse_transform(y)
oof_score = balanced_accuracy_score(oof_true_labels, oof_preds_labels)

print(f'OOF balanced accuracy : {oof_score:.5f}')
print(f'Baseline OOF          : {BASELINE_OOF:.5f}')
delta = oof_score - BASELINE_OOF
print(f'Delta vs baseline     : {"+" if delta >= 0 else ""}{delta:.5f}  {"✓ improvement" if delta > 0 else "✗ no improvement"}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

classes_ordered = ['GALAXY', 'QSO', 'STAR']

cm = confusion_matrix(oof_true_labels, oof_preds_labels, labels=classes_ordered)
ConfusionMatrixDisplay(cm, display_labels=classes_ordered).plot(cmap='Blues')
plt.title(f'LightGBM v2 OOF — Balanced Acc: {oof_score:.4f}')
plt.tight_layout()

In [ ]:
# Per-class breakdown — focus on STAR precision vs baseline (was 0.89)
print(classification_report(oof_true_labels, oof_preds_labels, target_names=classes_ordered))

## 8. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
importance_df.plot.barh(x='feature', y='importance', ax=ax, legend=False)
ax.set_title('LightGBM v2 Feature Importance (last fold)')
ax.invert_yaxis()
plt.tight_layout()

print(importance_df.to_string(index=False))

## 9. Generate Submission

In [ ]:
test_pred_labels = le.inverse_transform(test_proba.argmax(axis=1))

submission = pd.DataFrame({'id': test.index, 'class': test_pred_labels})
submission.to_csv('submission_lgbm_v2.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(submission['class'].value_counts())
submission.head()

In [ ]:
from IPython.display import FileLink, display

display(FileLink('submission_lgbm_v2.csv'))

## 10. Results

| Model | OOF | LB | Delta OOF | Notes |
|---|---|---|---|---|
| LightGBM baseline | 0.96418 | 0.96509 | — | v1 features, default params |
| LightGBM v2 | ... | ... | ... | + log_redshift, interactions, Optuna |

**STAR precision vs baseline (was 0.89):**  
**Best Optuna params:**  
**Which v2 features were most impactful (by importance rank):**